In [0]:
import pyspark.sql.functions as f

In [0]:
df_hospital_a = spark.read.parquet("/mnt/bronze/hospital-a/departments")

In [0]:
df_hospital_b = spark.read.parquet("/mnt/bronze/hospital-b/departments")

In [0]:
df_merged = df_hospital_a.unionByName(df_hospital_b)

In [0]:
df_merged = df_merged.withColumn("src_dept_id", f.col('deptid')) \
                    .withColumn("dept_id", f.concat(f.col('deptid'),f.lit('-'),f.col('datasource'))) \
                        .drop('deptid')

In [0]:
df_merged.createOrReplaceTempView("departments")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.departments(
  dept_id STRING,
  src_dept_id STRING,
  name STRING,
  datasource STRING,
  is_quarantined BOOLEAN
)
USING DELTA

In [0]:
%sql
INSERT INTO silver.departments
SELECT
  dept_id,
  src_dept_id,
  name,
  datasource,
  CASE 
    WHEN src_dept_id IS NULL OR name IS NULL THEN true
    ELSE FALSE
  END as is_quarantined
FROM departments

In [0]:
%sql
SELECT * FROM silver.departments